In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
FEEDBACK_FILE = "feedback_day1.csv"

LIKERT = ["Do not agree with", "Rather do not agree with", "Rather agree with", "Agree with"]
# diverging: red for disagreement, blue for agreement, darker = stronger
LIKERT_COLORS = ["#b8302f", "#ee7f7e", "#6da7ec", "#1c5cab"]
BAR_COLOR = "#2a78d6"

# answer order for the other closed questions (best/most first)
ORDERS = {
    "Overall: How satisfied were you with the course?":
        ["Excellent", "Very Good", "Good", "Satisfactory"],
    "Recommendation: Would you recommend the course?": ["Yes", "No"],
    "Balance of theoretical and practical content: The balance of theoretical and practical content was...":
        ["Too theoretical", "About right", "Too practical"],
    "Tool(s)/Resource(s) before the Course: Have you used the tool(s)/resource(s) covered in the course before?":
        ["Frequently (weekly to daily)", "Occasionally (once in a while to monthly)",
         "Never - used other service", "Never - aware of them, but not used them"],
    "Tool(s)/Resource(s) after the Course: Will you use the tool(s)/resource(s) covered in the course again?":
        ["Yes", "Maybe", "No"],
    "Course advertisement: Where did you see the course advertised?": None,
}

In [ ]:
df = pd.read_csv(FEEDBACK_FILE)
# drop identifying columns: submitter fields are empty, follow-up contact details are not needed here
df = df.drop(columns=[c for c in df.columns if c.startswith("Submitter") or "follow-up" in c])
SESSION = next(c for c in df.columns if c.startswith("About yourself: Which"))
print(f"{len(df)} responses")

In [ ]:
df.head()

In [ ]:
def bar(ax, counts, title):
    counts = counts[::-1]  # first entry on top
    ax.barh(counts.index, counts.values, color=BAR_COLOR, height=0.7)
    for y, value in enumerate(counts.values):
        ax.text(value, y, f" {value}", va="center", fontsize=9)
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlim(0, counts.max() * 1.1)  # room for the count labels
    ax.set_xticks([])
    ax.spines[["top", "right", "bottom"]].set_visible(False)


def answer_counts(column, order=None):
    counts = df[column].value_counts()
    if order:
        # keep the scale order, show unused options as 0, append unexpected answers
        counts = counts.reindex(order + [a for a in counts.index if a not in order], fill_value=0)
    return counts

In [ ]:
# who answered
fig, axes = plt.subplots(3, 1, figsize=(8, 7))
bar(axes[0], answer_counts(SESSION).rename(lambda s: s.split(": ", 1)[-1]), "Session")
bar(axes[1], answer_counts("About yourself: What is your background?"), "Background")
bar(axes[2], answer_counts("About yourself: In which field are you working?"), "Field")
fig.tight_layout()
plt.savefig("feedback_who_day1.png", dpi=300)
plt.show()

In [ ]:
# all statements answered on the agree/disagree scale
LIKERT_ITEMS = [c for c in df.columns if df[c].dropna().isin(LIKERT).all() and df[c].notna().any()]


SECTION_LABELS = {"General conditions during the workshop": "Conditions", "Your feedback": "Overall"}


def item_label(column):
    section, statement = column.split(": ", 1)
    section = section.rstrip(".").removeprefix("In this ")
    return f"{SECTION_LABELS.get(section, section)}: {statement.lstrip('.')}"


likert = pd.DataFrame(
    {item_label(c): df[c].value_counts().reindex(LIKERT, fill_value=0) for c in LIKERT_ITEMS}
).T
likert

In [ ]:
def likert_chart(likert):
    fig, ax = plt.subplots(figsize=(11, 0.4 * len(likert) + 1.5))
    y = range(len(likert))[::-1]  # first statement on top
    # disagreement extends left of zero, agreement right of zero
    left = -(likert[LIKERT[0]] + likert[LIKERT[1]])
    for answer, color in zip(LIKERT, LIKERT_COLORS):
        values = likert[answer]
        ax.barh(y, values, left=left, color=color, height=0.75, edgecolor="white", linewidth=1, label=answer)
        for yi, l, v in zip(y, left, values):
            if v > 0:
                text_color = "white" if color in (LIKERT_COLORS[0], LIKERT_COLORS[3]) else "black"
                ax.text(l + v / 2, yi, v, ha="center", va="center", fontsize=8, color=text_color)
        left = left + values
    ax.axvline(0, color="#52514e", linewidth=0.8)
    # symmetric range so the zero line sits in the middle
    ax.set_xlim(-len(df), len(df))
    ax.set_yticks(list(y), likert.index, fontsize=9)
    ax.set_xticks([])
    ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
    ax.legend(ncols=4, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


fig = likert_chart(likert)
fig.savefig(FEEDBACK_FILE.replace(".csv", "_likert.png"), bbox_inches="tight")
plt.show()

In [ ]:
# share of (rather) agreeing answers per statement, lowest first: where to improve
summary = pd.DataFrame({
    "agree": likert[LIKERT[2:]].sum(axis=1),
    "disagree": likert[LIKERT[:2]].sum(axis=1),
})
summary["agree %"] = (100 * summary["agree"] / (summary["agree"] + summary["disagree"])).round()
summary.sort_values("agree %")

In [ ]:
# mean score per session, 1 = do not agree ... 4 = agree
scores = df[LIKERT_ITEMS].apply(lambda c: c.map({a: i + 1 for i, a in enumerate(LIKERT)}))
scores.columns = [item_label(c) for c in LIKERT_ITEMS]
scores.groupby(df[SESSION].str.split(": ", n=1).str[-1]).mean().T.round(1)

In [ ]:
# raw answer counts per session for key questions; few responses per session, so no percentages
KEY_QUESTIONS = [
    "In this Workshop...: ...the lecturers made good use of the time available",
    "Hands-on exercises...: ...were achievable in the given amount of time",
    "Overall: How satisfied were you with the course?",
    "Recommendation: Would you recommend the course?",
]
session = df[SESSION].str.replace(r"Day \d: ", "", regex=True).rename("session")

for column in KEY_QUESTIONS:
    table = pd.crosstab(session, df[column])
    order = ORDERS.get(column, LIKERT)
    table = table.reindex(columns=order + [a for a in table.columns if a not in order], fill_value=0)
    table["n"] = table.sum(axis=1)
    table.columns.name = column.split(": ", 1)[-1]
    display(table)

In [ ]:
# remaining closed questions
counts = {column: answer_counts(column, order) for column, order in ORDERS.items()}
# panel height follows the number of answer options, so all bars have the same thickness
heights = [len(c) + 1 for c in counts.values()]
fig, axes = plt.subplots(len(counts), 1, figsize=(8, 0.35 * sum(heights)), height_ratios=heights)
for ax, (column, column_counts) in zip(axes, counts.items()):
    bar(ax, column_counts, column.split(": ", 1)[-1])
fig.tight_layout()
fig.savefig(FEEDBACK_FILE.replace(".csv", "_questions.png"), bbox_inches="tight")
plt.show()

In [ ]:
# free-text answers, grouped by session
for column in [c for c in df.columns if c.startswith("Your feedback:") and c not in LIKERT_ITEMS]:
    print(f"## {column.split(': ', 1)[-1]}\n")
    answers = pd.DataFrame({"session": session, "answer": df[column]}).dropna().sort_values("session")
    for s, answer in zip(answers["session"], answers["answer"]):
        print(f"- [{s}] {answer}")
    print()